In [ ]:
"""
First spin up vLLM server with:

vllm serve Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4 \
  --quantization gptq_marlin \
  --dtype auto \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.85 \
  --max-num-seqs 16
  
Can profile with:

vllm bench serve \
  --backend vllm \
  --model Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4 \
  --dataset-name random \
  --num-prompts 50 \
  --request-rate 10 \
  --result-filename metrics.json
  
"""

In [1]:
import os
# Force vLLM to use spawn method to avoid CUDA fork errors in Jupyter
#os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Tell the C++ compiler's linker exactly where to find Conda's CUDA libraries
#conda_prefix = os.environ.get("CONDA_PREFIX", "/home/dylan/miniconda3")
#os.environ["LIBRARY_PATH"] = f"{conda_prefix}/lib:" + os.environ.get("LIBRARY_PATH", "")

In [2]:
from experiments_vllm import *
from data import *
from transformers import AutoTokenizer

In [3]:
# Set seeds
seed = 43
np.random.seed(seed);
torch.manual_seed(seed);

In [ ]:
# Initialize the Async client pointing to the local vLLM server
client = AsyncOpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

In [ ]:
# Configuration
# For full dataset: n_samples = 15000, max_new_tokens = 100, batch_size = 16
model = "Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4"
n_samples = 100
max_new_tokens = 100
sampling_params = SamplingParams(max_tokens=max_new_tokens)
batch_size = 16
max_context_len = 1024
n_warmup_samples = 2

In [ ]:
# Select prompts
sample_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a python script to scrape a website.",
    "What are the benefits of MoE (Mixture of Experts) architectures?",
    "Tell me a short sci-fi story about a sentient coffee machine.",
    "Summarize the history of the Roman Empire in 3 paragraphs."
]

In [4]:
# Run prompts on vLLM server asynchronously
asyncio.run(run_batch(sample_prompts, print_output=True))


Initializing vLLM...
INFO 04-22 15:05:08 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'max_num_seqs': 16, 'disable_log_stats': True, 'model': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4'}
WARNING 04-22 15:05:08 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1


INFO 04-22 15:05:09 [model.py:549] Resolved architecture: Qwen2MoeForCausalLM
INFO 04-22 15:05:09 [model.py:1678] Using max model len 4096
INFO 04-22 15:05:10 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 04-22 15:05:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-22 15:05:10 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=834432) INFO 04-22 15:05:19 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', speculative_config=None, tokenizer='Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_

(EngineCore pid=834432) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=834432) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:02<00:04,  2.14s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:04<00:02,  2.21s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:04<00:00,  1.29s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:04<00:00,  1.53s/it]
(EngineCore pid=834432) 


(EngineCore pid=834432) INFO 04-22 15:05:27 [default_loader.py:384] Loading weights took 4.61 seconds
(EngineCore pid=834432) INFO 04-22 15:05:28 [gpu_model_runner.py:4820] Model loading took 7.83 GiB memory and 6.756915 seconds
(EngineCore pid=834432) INFO 04-22 15:05:32 [backends.py:1051] Using cache directory: /home/dylan/.cache/vllm/torch_compile_cache/d0279df09c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=834432) INFO 04-22 15:05:32 [backends.py:1111] Dynamo bytecode transform time: 3.43 s
(EngineCore pid=834432) INFO 04-22 15:05:33 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.218 s
(EngineCore pid=834432) INFO 04-22 15:05:33 [decorators.py:303] Directly load AOT compilation from path /home/dylan/.cache/vllm/torch_compile_cache/torch_aot_compile/6a0121729d110e99793c55be7a0916976ef61a82b846da14503d0e81c2fa339b/rank_0_0/model
(EngineCore pid=834432) INFO 04-22 15:05:33 [monitor.py:48] torch.compile took 5.29 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 7/7 [00:00<00:00, 10.64it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 5/5 [00:00<00:00, 12.29it/s]


(EngineCore pid=834432) INFO 04-22 15:05:39 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.17 GiB
(EngineCore pid=834432) INFO 04-22 15:05:39 [gpu_worker.py:597] CUDA graph pool memory: 0.17 GiB (actual), 0.54 GiB (estimated), difference: 0.37 GiB (222.1%).
(EngineCore pid=834432) INFO 04-22 15:05:39 [core.py:283] init engine (profile, create kv cache, warmup model) took 10.96 seconds


(EngineCore pid=834432) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=834432) INFO 04-22 15:05:40 [vllm.py:790] Asynchronous scheduling is enabled.
